# Section 2.8 Encoding Word Positions

[Token Embeddings](https://www.youtube.com/watch?v=ghCSGRgVB_o)

In [1]:
import torch
from torch.utils.data import Dataset

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            # Keep Dataset tensors on CPU. Move batches to the GPU after DataLoader yields them.
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


In [2]:
from torch.utils.data import DataLoader
import tiktoken

# Reuse this in later modules instead of scattering `.cuda()` calls.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Call this immediately after reading each batch in a training or inference loop.
def move_batch_to_device(batch, device):
    input_ids, target_ids = batch
    return input_ids.to(device, non_blocking=True), target_ids.to(device, non_blocking=True)

def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0, pin_memory=False):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    return dataloader


In [3]:
file_path = 'the-verdict.txt'
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False, pin_memory=device.type == "cuda")
data_iter = iter(dataloader)
inputs, targets = move_batch_to_device(next(data_iter), device)
print("Token IDs:
", inputs)
print("
Inputs shape:
", inputs.shape)


Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


Previously, we focused on very small embedding sizes for simplicity. Now, let’s consider more realistic and useful embedding sizes and encode the input tokens into a 256-dimensional vector representation, which is smaller than what the original GPT-3 model used (in GPT-3, the embedding size is 12,288 dimensions) but still reasonable for experimentation. Furthermore, we assume that the token IDs were created by the BPE tokenizer we implemented earlier, which has a vocabulary size of 50,257, which is the voabulary size of GPT-2. So the 'tokenizer' object above from `tiktoken.get_encoding("gpt-2")` returns a tokenizer with 

In [4]:
vocab_size = 50257
output_dim = 256

# Move modules to the selected device so their weights match GPU input tensors.
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim).to(device)


In [5]:
# inputs already moved to device when read from the DataLoader.
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)


torch.Size([8, 4, 256])


The concept of the below code block is explained more fully [here](https://youtu.be/ghCSGRgVB_o?t=1652). For a GPT model’s absolute embedding approach, we just need to create another embedding layer that has the same embedding dimension as the token_embedding_ layer:

In [6]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim).to(device)

# For tensors that index GPU modules, create them on the same device.
pos_embeddings = pos_embedding_layer(torch.arange(context_length, device=device))
print(pos_embeddings.shape)


torch.Size([4, 256])


The input to the pos_embeddings is usually a placeholder vector torch.arange(context_ length), which contains a sequence of numbers 0, 1, ..., up to the maximum input length –1. The context_length is a variable that represents the supported input size of the LLM. Here, we choose it similar to the maximum length of the input text.  In practice, input text can be longer than the supported context length, in which case we have to truncate the text.

As we can see, the positional embedding tensor consists of four 256-dimensional vectors.  We can now add these directly to the token embeddings, where PyTorch will add the 4 × 256–dimensional pos_embeddings tensor to each 4 × 256–dimensional token embedding tensor in each of the eight batches:

In [7]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
